In [ ]:
import copernicusmarine

In [ ]:
%%time
ds_temp = copernicusmarine.open_dataset(dataset_id='cmems_mod_med_phy-temp_my_4.2km_P1D-m')
ds_sal  = copernicusmarine.open_dataset(dataset_id='cmems_mod_med_phy-sal_my_4.2km_P1D-m')

In [ ]:
import hvplot.xarray
import panel as pn
import numpy as np
pn.extension()

In [ ]:
%%time
# Gulf of Lion, surface, 2026-03-31
T_surf = (
    ds_temp['thetao']
    .sel(time='2026-03-31', method='nearest')
    .sel(depth=0, method='nearest')
    .sel(longitude=slice(2.0, 6.5), latitude=slice(41.0, 44.5))
    .load()
)
S_surf = (
    ds_sal['so']
    .sel(time='2026-03-31', method='nearest')
    .sel(depth=0, method='nearest')
    .sel(longitude=slice(2.0, 6.5), latitude=slice(41.0, 44.5))
    .load()
)
print(f"T: {float(T_surf.min()):.2f} – {float(T_surf.max()):.2f} °C")
print(f"S: {float(S_surf.min()):.3f} – {float(S_surf.max()):.3f} PSU")

In [ ]:
def sigma_t(T, S):
    # Potential density anomaly σ_t (kg/m³) — UNESCO EOS-80 at p=0
    rho_w = (999.842594
             + 6.793952e-2  * T
             - 9.095290e-3  * T**2
             + 1.001685e-4  * T**3
             - 1.120083e-6  * T**4
             + 6.536332e-9  * T**5)
    B = (8.24493e-1
         - 4.0899e-3 * T
         + 7.6438e-5 * T**2
         - 8.2467e-7 * T**3
         + 5.3875e-9 * T**4)
    C = -5.72466e-3 + 1.0227e-4 * T - 1.6546e-6 * T**2
    D = 4.8314e-4
    rho = rho_w + B * S + C * S**1.5 + D * S**2
    return rho - 1000.0

sig_surf = sigma_t(T_surf, S_surf)
sig_surf.name = 'sigma_t'
sig_surf.attrs = {'long_name': 'Potential density anomaly', 'units': 'kg/m³'}
print(f"σ_t: {float(np.nanmin(sig_surf)):.3f} – {float(np.nanmax(sig_surf)):.3f} kg/m³")
sig_surf

In [ ]:
import geoviews.feature as gf
import cartopy.feature as cfeature

raster   = sig_surf.hvplot(x='longitude', y='latitude', rasterize=True, geo=True,
                            cmap='viridis', clabel='σ_t (kg/m³)', tiles='ESRI',
                            title='Potential density anomaly — Gulf of Lion, 2026-03-31 (surface)')
contours = sig_surf.hvplot.contour(x='longitude', y='latitude', geo=True, levels=10,
                                    line_width=0.8, colorbar=True, cmap='gray')
raster * contours

In [ ]:
%%time
# 3D density — all depths, Gulf of Lion, 2026-03-31
T_3d = (
    ds_temp['thetao']
    .sel(time='2026-03-31', method='nearest')
    .sel(longitude=slice(2.0, 6.5), latitude=slice(41.0, 44.5))
    .load()
)
S_3d = (
    ds_sal['so']
    .sel(time='2026-03-31', method='nearest')
    .sel(longitude=slice(2.0, 6.5), latitude=slice(41.0, 44.5))
    .load()
)
sig_3d = sigma_t(T_3d, S_3d)
sig_3d.name = 'sigma_t'
sig_3d.attrs = {'long_name': 'Potential density anomaly', 'units': 'kg/m³'}
print(f"σ_t 3D: {float(np.nanmin(sig_3d)):.3f} – {float(np.nanmax(sig_3d)):.3f} kg/m³")
sig_3d

In [ ]:
vmin_3d = float(np.nanmin(sig_3d.values))
vmax_3d = float(np.nanmax(sig_3d.values))
print(f"sig_3d  →  min = {vmin_3d:.3f}   max = {vmax_3d:.3f} kg/m³")

sig_3d.hvplot(
    x='longitude', y='latitude',
    groupby='depth',
    rasterize=True, geo=True,
    cmap='viridis', tiles='ESRI',
    clabel='σ_t (kg/m³)',
    clim=(vmin_3d, vmax_3d),
    widget_location='bottom',
    title='Potential density anomaly — Gulf of Lion, 2026-03-31'
)